# Pilot (2018) — LandIQ tomato polygons × AlphaEarth embeddings (Earth Engine)

Downloads **all 64 embedding bands** (`A00`–`A63`, ~10 m) from the Google Earth Engine collection **`GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL`** for **`alpha_earth.gee.pilot_polygon_count`** polygons: use **`null` in `paths.local.yaml` for every tomato polygon** in the GPKG, an **integer** for the first *N* only, or omit the key for a quick default of **5**.

**LandIQ 2018 × AlphaEarth:** This pilot lives under **`earth_engine/years/2018/`** for **2018 survey polygons** and uses **`alpha_earth.gee.embedding_year`** (default **2018**) so embeddings match the **same calendar year** as LandIQ. The [GEE catalog](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_SATELLITE_EMBEDDING_V1_ANNUAL) includes 2018 (64 bands, 10 m). Other survey years: duplicate this folder pattern under `years/<YEAR>/` and align `configs/paths.local.yaml`.

**Earth Engine auth:** Credentials are read from `%USERPROFILE%\.config\earthengine\credentials` (set **`EARTHENGINE_CREDENTIALS`** to override). If `ee.Initialize()` still asks to authorize, run **`earthengine authenticate`** once in the same conda env, or let the next cell call **`ee.Authenticate()`** after a failure. Do **not** commit secrets.

**Exploration:** [leafmap AlphaEarth + MapLibre](https://leafmap.org/maplibre/AlphaEarth/) shows how to browse embeddings interactively (optional cell at the end).

**Speed / CPU:** Threaded workers each run **EE download then polygon→NaN mask** in the same thread (`download_workers`). **Resolution:** keep `scale_m: 10` for native AlphaEarth annual pixels.

**Outputs:** `data/derived/alpha_earth_clips/ee/landiq2018/<run_id>_…/` — one GeoTIFF per polygon + `manifest.json` (gitignored). Filenames: `poly_r00000_srcidx…_allbands_A00-A63.tif`. **Full-GPKG runs** are slow and quota-heavy; use a small integer `pilot_polygon_count` for smoke tests.

### Optional: install / upgrade packages in this kernel

```python
# %pip install -q -U earthengine-api geopandas pyyaml rasterio
# %pip install -q -U leafmap  # only if you run the optional MapLibre cell below
```

In [ ]:
from __future__ import annotations

import json
import os
from datetime import datetime, timezone
from pathlib import Path

import ee
import geopandas as gpd
import yaml

from src.alpha_earth.gee_embeddings import (
    DEFAULT_EMBEDDING_COLLECTION,
    NATIVE_ALPHA_EARTH_ANNUAL_SCALE_M,
    annual_embedding_mosaic,
    download_clipped_geotiff,
    embedding_band_names,
    shapely_to_ee_geometry,
    validate_embedding_year,
    warn_unless_native_embedding_scale,
    write_pilot_manifest,
)
from src.utils.paths import REPO_ROOT, landiq_tomato_gpkg_path, landiq_non_tomato_gpkg_path

cfg_path = REPO_ROOT / "configs" / "paths.local.yaml"
if not cfg_path.is_file():
    cfg_path = REPO_ROOT / "configs" / "paths.example.yaml"
cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))

dataset_label = cfg.get("landiq", {}).get("dataset_label", "tomato")
dataset_suffix = "_non_tomato" if dataset_label == "non_tomato" else ""

tomato_path = (
    landiq_non_tomato_gpkg_path(cfg, REPO_ROOT)
    if dataset_label == "non_tomato"
    else landiq_tomato_gpkg_path(cfg, REPO_ROOT)
)

if not tomato_path.is_file():
    raise FileNotFoundError(
        f"GPKG not found for dataset_label={dataset_label!r}: {tomato_path}. Run LandIQ filter notebook or CLI first."
    )

gdf = gpd.read_file(tomato_path)
if gdf.crs is None:
    raise ValueError("Tomato layer has no CRS; set CRS before export.")
gdf_wgs84 = gdf.to_crs(4326)

gee_cfg = cfg.get("alpha_earth", {}).get("gee", {})
_ppc = gee_cfg.get("pilot_polygon_count", 5)
if _ppc is None:
    subset = gdf_wgs84.copy()
else:
    n = int(_ppc)
    if n < 1:
        raise ValueError("alpha_earth.gee.pilot_polygon_count must be null (all) or a positive integer.")
    subset = gdf_wgs84.head(n).copy()

print("Tomato GPKG:", tomato_path.resolve())
print("Total polygons:", len(gdf), "| exporting:", len(subset), end="")
print(" (all)" if _ppc is None else f" (pilot_polygon_count={_ppc})")
print("WGS84 bounds (subset):", subset.total_bounds)

In [ ]:
# Earth Engine: load OAuth JSON and pass credentials explicitly (avoids flaky get_persistent_credentials).
# File: %USERPROFILE%\.config\earthengine\credentials — or set EARTHENGINE_CREDENTIALS to full path.

from typing import Optional, Tuple

from google.oauth2.credentials import Credentials as GoogleCredentials


def _ee_credentials_path() -> Path:
    env = os.environ.get("EARTHENGINE_CREDENTIALS")
    if env:
        return Path(env)
    return Path.home() / ".config" / "earthengine" / "credentials"


def _oauth_from_ee_file(path: Path) -> Tuple[Optional[GoogleCredentials], Optional[str]]:
    if not path.is_file():
        return None, None
    try:
        raw = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None, None
    cid, secret, rt = raw.get("client_id"), raw.get("client_secret"), raw.get("refresh_token")
    if not (cid and secret and rt):
        return None, None
    creds = GoogleCredentials(
        None,
        refresh_token=rt,
        token_uri="https://oauth2.googleapis.com/token",
        client_id=cid,
        client_secret=secret,
        scopes=raw.get("scopes"),
    )
    proj = raw.get("project")
    return creds, (str(proj) if proj is not None else None)


cred_path = _ee_credentials_path()
oauth_creds, project_in_file = _oauth_from_ee_file(cred_path)

ee_project = (
    os.environ.get("EE_PROJECT")
    or os.environ.get("GOOGLE_CLOUD_PROJECT")
    or os.environ.get("GCP_PROJECT")
    or gee_cfg.get("project")
    or project_in_file
)

print("Credentials file:", cred_path.resolve(), "| exists:", cred_path.is_file())
if cred_path.is_file():
    print("  size (bytes):", cred_path.stat().st_size, "| oauth keys ok:", oauth_creds is not None)


def _init_ee() -> None:
    if oauth_creds is not None:
        try:
            if ee_project:
                ee.Initialize(credentials=oauth_creds, project=str(ee_project))
            else:
                ee.Initialize(credentials=oauth_creds)
            return
        except Exception as ex:
            print("ee.Initialize(credentials=...) failed:", repr(ex))

    try:
        if ee_project:
            ee.Initialize(project=str(ee_project))
        else:
            ee.Initialize()
    except ee.EEException as ex:
        low = str(ex).lower()
        if "authorize" in low or "authenticate" in low:
            print("Opening browser for one-time ee.Authenticate()...")
            ee.Authenticate()
            if ee_project:
                ee.Initialize(project=str(ee_project))
            else:
                ee.Initialize()
        else:
            raise


_init_ee()
print("Earth Engine initialized.")
print("  project:", ee_project or "(default)")
try:
    print("  ee.data.getAssetRoots():", ee.data.getAssetRoots())
except Exception as e:
    print("  (asset roots unavailable)", e)

In [ ]:
landiq_year = cfg.get("landiq", {}).get("year")
embedding_year = int(gee_cfg.get("embedding_year", 2018))
collection_id = gee_cfg.get("embedding_collection", DEFAULT_EMBEDDING_COLLECTION)
scale_m = int(gee_cfg.get("scale_m", 10))
export_crs = gee_cfg.get("export_crs", "EPSG:4326")

validate_embedding_year(embedding_year)
warn_unless_native_embedding_scale(scale_m, collection_id)
print(
    "EE export scale:", scale_m, "m | catalog native (AlphaEarth annual):",
    NATIVE_ALPHA_EARTH_ANNUAL_SCALE_M,
    "m — keep scale_m=10 for full native resolution (no EE resampling).",
)

run_tag = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ly = f"{landiq_year}" if landiq_year is not None else "unknown"
folder_name = f"{run_tag}_landiq{ly}{dataset_suffix}_alphaearth{embedding_year}_n{len(subset)}"
clips_root = REPO_ROOT / cfg["data"]["alpha_earth_clips"]
run_dir = clips_root / "ee" / f"landiq{ly}{dataset_suffix}" / folder_name
run_dir.mkdir(parents=True, exist_ok=True)

print("GEE collection:", collection_id)
print("Embedding calendar year:", embedding_year, "| LandIQ survey year (from config):", landiq_year)
print("Export directory:", run_dir.resolve())

In [ ]:
# Preflight: build one global-year mosaic (no per-polygon getInfo); confirm 64 bands
embedding_base = annual_embedding_mosaic(embedding_year, collection_id)
first_row = subset.iloc[0]
first_geom = shapely_to_ee_geometry(first_row.geometry)
pre_img = embedding_base.clip(first_geom)
band_list = pre_img.bandNames().getInfo()
print("Band count:", len(band_list))
print("First / last bands:", band_list[:3], "...", band_list[-3:])
expected = embedding_band_names()
assert band_list == expected, f"Unexpected bands: got {len(band_list)}, expected {len(expected)}"

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def _row_index_as_json(i):
    try:
        return int(i)
    except (TypeError, ValueError):
        return str(i)


def _download_one(task):
    rank, src_idx, row = task
    geom_ee = shapely_to_ee_geometry(row.geometry)
    fname = f"poly_r{rank:05d}_srcidx{src_idx}_allbands_A00-A63.tif"
    out_path = run_dir / fname
    # Download + polygon→NaN in the same thread (matches the original working path).
    download_clipped_geotiff(
        embedding_base,
        geom_ee,
        out_path,
        scale_m=scale_m,
        crs=export_crs,
        polygon_wgs84=row.geometry,
        rasterize_all_touched=True,
    )
    return rank, _row_index_as_json(src_idx), str(out_path.resolve())


_dw = int(gee_cfg.get("download_workers", 8))
max_workers = max(1, min(_dw, 32))
tasks = [(rank, src_idx, row) for rank, (src_idx, row) in enumerate(subset.iterrows())]
print(
    f"Parallel download + polygon NaN mask per file (threads, scale_m={scale_m} m): "
    f"{max_workers} workers (alpha_earth.gee.download_workers={_dw})."
)

out_by_rank: dict[int, str] = {}
src_by_rank: dict[int, int | str] = {}
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(_download_one, t) for t in tasks]
    for fut in as_completed(futures):
        rank, src_j, path = fut.result()
        out_by_rank[rank] = path
        src_by_rank[rank] = src_j
        print(f"Done rank {rank} ({Path(path).name})")

_order = sorted(out_by_rank)
out_files = [out_by_rank[r] for r in _order]
src_indices = [src_by_rank[r] for r in _order]

manifest_path = run_dir / "manifest.json"
write_pilot_manifest(
    manifest_path,
    source_gpkg=str(tomato_path.resolve()),
    landiq_survey_year=int(landiq_year) if landiq_year is not None else None,
    embedding_year=embedding_year,
    collection_id=collection_id,
    scale_m=scale_m,
    crs_export=export_crs,
    polygon_indices=src_indices,
    output_files=out_files,
    band_names=embedding_band_names(),
    polygon_mask_all_touched=True,
)

print("Done.")
print("Manifest:", manifest_path.resolve())
print(json.dumps(json.loads(manifest_path.read_text(encoding="utf-8")), indent=2)[:2000], "...")

In [ ]:
# Resume helper: detect existing outputs in run_dir and download only missing polygons.
# Run this after the main download cell if a long run was interrupted.

from concurrent.futures import ThreadPoolExecutor, as_completed


def _expected_output(rank, src_idx):
    fname = f"poly_r{rank:05d}_srcidx{src_idx}_allbands_A00-A63.tif"
    return run_dir / fname


# Build full expected task list from current subset.
all_tasks = [(rank, src_idx, row) for rank, (src_idx, row) in enumerate(subset.iterrows())]

missing_tasks = []
for rank, src_idx, row in all_tasks:
    out_path = _expected_output(rank, src_idx)
    if (not out_path.exists()) or out_path.stat().st_size == 0:
        missing_tasks.append((rank, src_idx, row))

print(f"Run directory: {run_dir}")
print(f"Expected files: {len(all_tasks)}")
print(f"Already present: {len(all_tasks) - len(missing_tasks)}")
print(f"Missing: {len(missing_tasks)}")

if missing_tasks:
    _dw = int(gee_cfg.get("download_workers", 8))
    max_workers = max(1, min(_dw, 32))
    print(f"Resuming with {max_workers} download workers...")

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(_download_one, t) for t in missing_tasks]
        for fut in as_completed(futures):
            rank, src_j, path = fut.result()
            print(f"Recovered rank {rank} ({Path(path).name})")

# Rebuild manifest from files currently on disk (complete + resumed).
out_files = []
src_indices = []
for rank, (src_idx, _row) in enumerate(subset.iterrows()):
    out_path = _expected_output(rank, src_idx)
    if out_path.exists() and out_path.stat().st_size > 0:
        out_files.append(str(out_path.resolve()))
        src_indices.append(_row_index_as_json(src_idx))

manifest_path = run_dir / "manifest.json"
write_pilot_manifest(
    manifest_path,
    source_gpkg=str(tomato_path.resolve()),
    landiq_survey_year=int(landiq_year) if landiq_year is not None else None,
    embedding_year=embedding_year,
    collection_id=collection_id,
    scale_m=scale_m,
    crs_export=export_crs,
    polygon_indices=src_indices,
    output_files=out_files,
    band_names=embedding_band_names(),
    polygon_mask_all_touched=True,
)

print("Resume check complete.")
print(f"Files now present: {len(out_files)} / {len(all_tasks)}")
print("Manifest:", manifest_path.resolve())

### Quick local QC (GeoTIFF shape / band count)

Confirms the first download has 64 bands and a non-empty raster window.

In [ ]:
import rasterio

first_tif = sorted(run_dir.glob("poly_r*_allbands_A00-A63.tif"))[0]
with rasterio.open(first_tif) as ds:
    print("File:", first_tif.name)
    print("Shape (bands, height, width):", ds.count, ds.height, ds.width)
    print("CRS:", ds.crs)
    print("Dtypes:", ds.dtypes[:5], "...", ds.dtypes[-1:])

### Optional — interactive MapLibre map (leafmap)

Same idea as the [leafmap AlphaEarth example](https://leafmap.org/maplibre/AlphaEarth/): browse embeddings over a basemap. Uncomment `leafmap` install in the setup cell if needed.

Below: center on the **first pilot polygon** and add a false-color embedding layer for **`embedding_year`** (three bands only — not all 64).

In [ ]:
# import leafmap.maplibregl as leafmap

# center = subset.iloc[0].geometry.centroid
# m = leafmap.Map(projection="globe", sidebar_visible=True)
# m.add_basemap("USGS.Imagery")
# m.add_alphaearth_gui()
# m.set_center(float(center.x), float(center.y), zoom=12)
# vis = {"min": -0.3, "max": 0.3, "bands": ["A01", "A16", "A09"]}
# m.add_ee_layer(pre_img, vis, name=f"Embeddings {embedding_year} (RGB preview)")
# m